# HW2 — Partitions, conditional entropy, and 1-WL on toy graphs

**Reading.** `handout.md` Q1–Q5; [`PAPER-ARXIV.md`](../../../../PAPER-ARXIV.md) §3.1 (Definitions 3.1–3.2, **Lemma 3.1**) and §3.3 (scope: $\mathcal{F}_\mathrm{WL}$).

**Goal.** Hand-derive $H(Y\mid\Pi)$ on a 6-node toy, code `cond_entropy`, prove **Lemma 3.1** (*purity ⟺ zero conditional entropy*) numerically, implement one round of 1-WL refinement, and exhibit the $C_6$ vs $2K_3$ blind spot — the **qualitative ceiling** of Xu et al. (ICLR 2019, GIN) and Morris et al. (AAAI 2019, $k$-GNNs) that **HW3's bracket will quantify**. Six assertion gates.

**Julia companion (optional).** [`julia-theory/notebooks/04_bayes_error_landscape.jl`](../../../julia-theory/notebooks/04_bayes_error_landscape.jl) is the reactive slider twin of Q2 (per-cell $e_C$, $q_C$ on a 3-simplex).

## Setup

Resolve `onboarding/projects/` on `sys.path`, attach the reflection log, and import the canonical references (used only for spot-checks; the student version derives its own implementations).

In [ ]:
import os, sys
from pathlib import Path
_here = Path(os.getcwd()).resolve()
for _p in [_here, *_here.parents]:
    if (_p / 'shared' / '__init__.py').exists() and _p.name == 'projects':
        _PROJECTS = _p
        break
else:
    raise RuntimeError('could not locate onboarding/projects/')
_REPO = _PROJECTS.parent.parent
if str(_REPO) not in sys.path:
    sys.path.insert(0, str(_REPO))

import numpy as np
import matplotlib
matplotlib.use('Agg')   # headless for CI
import matplotlib.pyplot as plt

from onboarding.projects import reflect
reflect.start(notebook='hw2', store=_PROJECTS / '.reflection.jsonl')
print(f'[setup] projects root: {_PROJECTS}')


In [ ]:
# Reuse hbin from HW1's solution if exported, else re-define inline.
from math import log2
def hbin(p: float) -> float:
    return 0.0 if (p <= 0 or p >= 1) else -(p*log2(p) + (1-p)*log2(1-p))


## Q1 — Hand-compute on a 6-node toy

$y = (0,0,1,1,1,0)$, $\Pi = \{\{0,1,2\}, \{3,4\}, \{5\}\}$.

By hand: $q = (1/2, 1/3, 1/6)$, $e = (1/3, 0, 0)$, $\varepsilon(\Pi) = 1/6 \approx 0.1667$, $H(Y\mid\Pi) = \tfrac{1}{2} H_\mathrm{bin}(1/3) \approx 0.4591$.

In [ ]:
y = np.array([0,0,1,1,1,0])
Pi = [np.array([0,1,2]), np.array([3,4]), np.array([5])]
n = sum(len(c) for c in Pi)
q = [len(c)/n for c in Pi]
e = []
for c in Pi:
    lbls = y[c]
    vals, counts = np.unique(lbls, return_counts=True)
    e.append(1 - counts.max()/len(c))
eps = sum(qi*ei for qi, ei in zip(q, e))
H = sum(qi*hbin(ei) for qi, ei in zip(q, e))
print(f'q={q}, e={e}, eps={eps:.4f}, H(Y|Pi)={H:.4f}')


**Gate Q1.** Hand and code must agree to $10^{-12}$.

In [ ]:
assert abs(q[0]-1/2) < 1e-12 and abs(q[1]-1/3) < 1e-12 and abs(q[2]-1/6) < 1e-12
assert abs(e[0]-1/3) < 1e-12 and e[1]==0 and e[2]==0
assert abs(eps - 1/6) < 1e-12, f'eps {eps} ≠ 1/6'
assert abs(H - 0.5*hbin(1/3)) < 1e-12, f'H {H} ≠ 0.5·hbin(1/3)'
print(f'[GATE OK] Q1: q,e,eps,H match hand computation; H={H:.6f}')


In [ ]:
reflect.log('Q2.hw2.Q1_hand', f'H(Y|Pi) on 6-node toy ≈ {H:.4f}', 'HIGH')


## Q2 — `cond_entropy(partition, labels)`

**Concept.** Sum cell-wise $q_C \cdot H_\mathrm{bin}(e_C)$. Edge cases: every cell singleton ⇒ $H = 0$. Single all-of-$\{0,\dots,n-1\}$ cell ⇒ $H = H_\mathrm{bin}(\varepsilon)$ (global Bayes error).

In [ ]:
def cond_entropy(partition, labels):
    """H(Y|Π) in bits."""
    n = sum(len(c) for c in partition)
    total = 0.0
    for c in partition:
        q_C = len(c) / n
        lbls = labels[c]
        _, counts = np.unique(lbls, return_counts=True)
        e_C = 1 - counts.max() / len(c)
        total += q_C * hbin(e_C)
    return total


**Distinguish — partition-entropy vs label-entropy.** $H(Y\mid\Pi)$ is NOT $H(Y)$; it's $H$ of the *Bayes error inside each cell*. Singletons drive $H$ to 0; the trivial 1-cell partition recovers $H_\mathrm{bin}(\varepsilon_{\mathrm{global}})$.

In [ ]:
# Singleton-cell partition: every cell has e=0 ⇒ H = 0.
sing = [np.array([i]) for i in range(len(y))]
H_sing = cond_entropy(sing, y)
# Trivial 1-cell partition: H = hbin(global Bayes error).
triv = [np.arange(len(y))]
H_triv = cond_entropy(triv, y)
_, counts = np.unique(y, return_counts=True)
eps_global = 1 - counts.max()/len(y)
print(f'H_sing={H_sing:.6f} H_triv={H_triv:.6f} hbin(eps_global)={hbin(eps_global):.6f}')
H_toy = cond_entropy(Pi, y)
print(f'H_toy={H_toy:.6f}')


**Gate Q2.** Singletons ⇒ 0; trivial ⇒ $H_\mathrm{bin}(\varepsilon)$; toy matches Q1.

In [ ]:
assert H_sing == 0.0, f'Q2: singleton H={H_sing} should be 0'
assert abs(H_triv - hbin(eps_global)) < 1e-12, f'Q2: triv {H_triv} ≠ hbin({eps_global})'
assert abs(H_toy - H) < 1e-12, f'Q2: toy {H_toy} ≠ hand {H}'
print(f'[GATE OK] Q2: cond_entropy matches edges and toy (H_toy={H_toy:.4f})')


In [ ]:
reflect.log('Q2.hw2.Q2_cond_entropy', 'cond_entropy correct on singleton, trivial, toy', 'HIGH')


## Q2.5 — **Lemma 3.1** (purity ⟺ zero conditional entropy)

**Statement (Paper §3.1).** Call $\Pi$ **pure for $Y$** if every cell $C \in \Pi$ is label-constant ($y$ is constant on $C$). Then

$$
H(Y \mid \Pi) = 0 \iff \Pi \text{ is pure for } Y.
$$

**Sketch.** $H(Y\mid\Pi) = \sum_C q_C H_\mathrm{bin}(e_C)$ with $q_C > 0$ and $H_\mathrm{bin}(e) \ge 0$, equality iff $e \in \{0, 1\}$. After per-cell binarisation $e_C \leftarrow \min(e_C, 1-e_C) \in [0, 1/2]$, the only zero is $e_C = 0$, i.e. cell is single-class.

**Adversarial framing.** A reader who only memorises Theorem 1 may conflate "low $H$" with "low $\varepsilon$". Lemma 3.1 is the *iff* that grounds the lower bracket: $H=0$ is **not** an accident of one cell, it is purity of the entire partition.

In [ ]:
def is_pure(partition, labels) -> bool:
    """True iff every cell is label-constant."""
    for C in partition:
        if len(set(labels[C].tolist())) > 1:
            return False
    return True


**Distinguish — *iff*, not just *if*.** A naive reader might prove only the easy direction (purity $\Rightarrow H=0$). The reverse — $H=0 \Rightarrow$ purity — uses strict positivity of $H_\mathrm{bin}$ on $(0, 1)$.

In [ ]:
# Build a non-trivial witness of each direction.
# (a) Pure partition: split y by label.
y_pure = np.array([0,0,0,1,1,1])
Pi_pure = [np.array([0,1,2]), np.array([3,4,5])]
H_pure  = cond_entropy(Pi_pure, y_pure)
print(f'pure: H={H_pure:.6f}, is_pure={is_pure(Pi_pure, y_pure)}')

# (b) Mixed cell ⇒ H>0.
y_mix = np.array([0,0,1,1,1,0])
Pi_mix = [np.array([0,1,2,3,4,5])]
H_mix  = cond_entropy(Pi_mix, y_mix)
print(f'mixed: H={H_mix:.6f}, is_pure={is_pure(Pi_mix, y_mix)}')

# (c) Border case: every singleton ⇒ trivially pure ⇒ H=0.
Pi_sing = [np.array([i]) for i in range(6)]
H_sing2 = cond_entropy(Pi_sing, y_mix)
print(f'singletons: H={H_sing2:.6f}, is_pure={is_pure(Pi_sing, y_mix)}')


**Gate Q2.5.** Forward (purity $\Rightarrow H=0$) AND reverse (mixed cell $\Rightarrow H>0$) on three witnesses.

In [ ]:
# Lemma 3.1 — forward direction: pure ⇒ H = 0.
assert is_pure(Pi_pure, y_pure) and H_pure == 0.0, f'Q2.5: pure partition has H={H_pure} ≠ 0'
assert is_pure(Pi_sing, y_mix) and H_sing2 == 0.0, f'Q2.5: singletons have H={H_sing2} ≠ 0'
# Reverse direction: H > 0 ⇒ NOT pure.
assert H_mix > 0 and not is_pure(Pi_mix, y_mix), f'Q2.5: mixed cell has H={H_mix} but is_pure={is_pure(Pi_mix, y_mix)}'
# Final: sweep 200 random partitions; the equivalence must never break.
import random
random.seed(1)
for trial in range(200):
    yy = np.array([random.randint(0, 1) for _ in range(8)])
    nc = random.randint(1, 4)
    perm = list(range(8)); random.shuffle(perm)
    cuts = sorted(random.sample(range(1, 8), nc - 1)) if nc > 1 else []
    edges = [0] + cuts + [8]
    PP = [np.array(perm[edges[i]:edges[i+1]]) for i in range(nc)]
    HH = cond_entropy(PP, yy)
    pp = is_pure(PP, yy)
    assert (HH == 0.0) == pp, f'Q2.5 random trial {trial}: H={HH}, is_pure={pp}'
print('[GATE OK] Q2.5: Lemma 3.1 verified on 3 hand witnesses + 200 random partitions')


In [ ]:
reflect.log('Q2.hw2.Q2.5_lemma_3_1',
            'Lemma 3.1 (purity ⟺ H(Y|Π)=0) verified on 3 witnesses + 200 random partitions',
            'HIGH')


## Q3+Q4 — One-round 1-WL refinement and the $C_6$ vs $2K_3$ blind spot

**Concept.** $\mathrm{color}_{t+1}(u) = \mathrm{hash}(\mathrm{color}_t(u), \{\!\!\{\mathrm{color}_t(v) : v \sim u\}\!\!\})$. Rename signatures densely each round. $C_6$ and $2K_3$ are both 2-regular and indistinguishable by 1-WL.

In [ ]:
def wl_step(edge_index, n, colors):
    """One round of 1-WL refinement on a directed-edge-index graph."""
    # Group neighbour colours per node.
    neighbours = [[] for _ in range(n)]
    src, dst = edge_index[0], edge_index[1]
    for u, v in zip(src, dst):
        neighbours[int(u)].append(int(colors[int(v)]))
    sigs = [(int(colors[u]), tuple(sorted(neighbours[u]))) for u in range(n)]
    table = {}
    new = np.zeros(n, dtype=int)
    for u, s in enumerate(sigs):
        if s not in table:
            table[s] = len(table)
        new[u] = table[s]
    return new


**Distinguish — 1-WL vs degree.** A 1-WL round subsumes degree (multiset size). Two regular graphs of the same degree (e.g. $C_6$ and $2K_3$) cannot be told apart even after stability.

In [ ]:
def edges_C6():
    pairs = [(i, (i+1) % 6) for i in range(6)]
    e = []
    for u, v in pairs:
        e += [(u, v), (v, u)]
    return np.array(e).T

def edges_2K3():
    pairs = [(0,1),(1,2),(0,2),(3,4),(4,5),(3,5)]
    e = []
    for u, v in pairs:
        e += [(u, v), (v, u)]
    return np.array(e).T

def wl_stabilise(edge_index, n, rounds=10):
    colors = np.zeros(n, dtype=int)
    for _ in range(rounds):
        new = wl_step(edge_index, n, colors)
        if np.array_equal(new, colors):
            break
        colors = new
    return colors

c6 = wl_stabilise(edges_C6(), 6)
k3 = wl_stabilise(edges_2K3(), 6)
def multiset(arr):
    _, cnts = np.unique(arr, return_counts=True)
    return tuple(sorted(cnts.tolist()))
print(f'C6 stable colors: {c6}  multiset {multiset(c6)}')
print(f'2K3 stable colors: {k3}  multiset {multiset(k3)}')


**Gate Q3+Q4.** $C_6$ and $2K_3$ stable multisets coincide; both equal $(6,)$.

In [ ]:
assert multiset(c6) == multiset(k3) == (6,), f'1-WL multisets {multiset(c6)} vs {multiset(k3)} — blind-spot test failed'
# Also check wl_step is non-trivial on the path P_3: endpoints split from middle.
p3 = np.array([[0,1,1,2],[1,0,2,1]])
col = np.zeros(3, dtype=int)
for _ in range(3):
    col = wl_step(p3, 3, col)
assert len(set(col.tolist())) == 2, f'P3 should split into 2 colour classes; got {col}'
assert col[0] == col[2] and col[0] != col[1], f'P3 endpoints {col[0]},{col[2]} vs middle {col[1]}'
print(f'[GATE OK] Q3+Q4: C6 and 2K3 indistinguishable by 1-WL; P3 endpoints split from middle')


In [ ]:
reflect.log('Q2.hw2.Q3Q4_wl_blindspot',
            'C6 vs 2K3 share stable-colour multiset (6,) under 1-WL; P3 separates endpoints from middle',
            'HIGH')


## Q4.5 — Provenance: Xu et al. (GIN) and Morris et al. ($k$-GNN)

The $C_6$ vs $2K_3$ blind spot you just exhibited is the central negative result of **two ICLR/AAAI 2019 papers**:

- **Xu, Hu, Leskovec, Jegelka.** *How Powerful are Graph Neural Networks?*   ICLR 2019. The **GIN** paper. Theorem 3: any MPNN with countable   multiset aggregator is at most as expressive as 1-WL.
- **Morris, Ritzert, Fey, Hamilton, Lenssen, Rattan, Grohe.** *Weisfeiler   and Leman Go Neural: Higher-order Graph Neural Networks.* AAAI 2019.   The **$k$-GNN** paper. Same qualitative ceiling at 1-WL; lifts it to   $k$-WL via tuple states.

Both papers say *qualitatively*: "MPNNs can never separate $C_6$ from $2K_3$." **Paper A** (Theorem 1, HW3 next week) sharpens this to a **quantitative budget**: for any MPNN-induced partition $\Pi$, the Bayes error obeys

$$
H_\mathrm{bin}^{-1}\!\big(H(Y\mid\Pi)\big) \;\le\; \varepsilon^*_\Pi \;\le\; \tfrac{1}{2}\, H(Y\mid\Pi).
$$

The slack at the worst point is **$w^\star \approx 0.1610$** attained at **$\varepsilon^\star = 1/5$** — the number HW3.Q4 will grid-search you to. *That* is the upgrade from "impossible" to "impossible by exactly this much."

In [ ]:
reflect.log('Q2.hw2.Q4.5_provenance',
            'Xu 2019 (GIN) + Morris 2019 (k-GNN) give the qualitative 1-WL ceiling; Theorem 1 quantifies it (w*≈0.1610).',
            'HIGH')


## Q5 — Writeup & calibration

`writeup.md` should mirror §Q1–§Q4. Below we roll up the reflection log.

In [ ]:
reflect.log('Q2.hw2.Q5_writeup',
            'Writeup mirrors hand-comp, cond_entropy, wl_step, blind-spot demonstration',
            'MEDIUM')
print('HW2 reflection log:')
for r in reflect.dump():
    if 'hw2' in r['notebook']:
        print(f"  [{r['level']:>10}] {r['concept']}: {r['claim']}")


**End of HW2.**